In [0]:
dbutils.widgets.text("my_sql_host","35.226.28.96")
my_sql_host=dbutils.widgets.get("my_sql_host")

dbutils.widgets.text("my_sql_port","3306")
my_sql_port=dbutils.widgets.get("my_sql_port")

dbutils.widgets.text("my_sql_user","root")
my_sql_user=dbutils.widgets.get("my_sql_user")

dbutils.widgets.text("my_sql_pwd","Admin@1234")
my_sql_pwd=dbutils.widgets.get("my_sql_pwd")

dbutils.widgets.text("my_sql_db","GCPMigrationmMeta")
my_sql_db=dbutils.widgets.get("my_sql_db")

dbutils.widgets.text("sak","/Volumes/workspace/default/gcptodbmigration/sk/datamigrationproject-494118-b1bc74441af6.json")
sak=dbutils.widgets.get("sak")

In [0]:
my_sql_host

In [0]:
%pip install mysql-connector-python

In [0]:
%pip install --upgrade google-cloud-storage
dbutils.library.restartPython()

In [0]:
%pip install google-cloud-bigquery
%pip install pyarrow

In [0]:
import json,os,datetime as dt
import mysql.connector as sql
from google.cloud import bigquery
from contextlib import contextmanager

mysql={
    "host":my_sql_host,
    "port":my_sql_port,
    "user":my_sql_user,
    "password":my_sql_pwd,
    "database":my_sql_db
}

mysql

In [0]:
@contextmanager
def mysql_conn():
    conn=sql.connect(host=mysql["host"],port=int(mysql["port"]),user=mysql["user"],password=mysql["password"],database=mysql["database"])

    try:
        yield conn
    except Exception as e:
        print(f"Error in mysql connection {e}")
    finally:
        conn.close()

In [0]:
def fetch_eligible_table():
    with mysql_conn() as conn:
        cur=conn.cursor(dictionary=True)
        cur.execute("select * from config_table where active_flag=1 and load_flag=1")
        rows=cur.fetchall()
        cur.close()
    return rows

fetch_eligible_table()

In [0]:
def bq_to_gcs_status(table_name,status,error=None):
    with mysql_conn() as conn:
        cur=conn.cursor()
        
        if status=="IN_PROGRESS":
            cur.execute("""update config_table set bq_to_gcs_status="IN_PROGRESS",last_run_ts=NOW(),error_message=NULL where table_name=%s""",(table_name,))

        elif status=="COMPLETED":
            cur.execute("""update config_table set bq_to_gcs_status="COMPLETED",last_run_ts=NOW(),error_message=NULL where table_name=%s""",(table_name,))

        else:
            cur.execute("""update config_table set bq_to_gcs_status="FAILED",last_run_ts=NOW(),error_message=%s where table_name=%s""",(str(err)[:2000] if err else "FAILED",table_name))

        conn.commit()
        cur.close()    

In [0]:
def bq_client():
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"]=sak
    with open(sak) as f:
        data=json.load(f).get("project_id")
    return bigquery.Client(project=data)

In [0]:
def export_table_to_gcs(source_project,source_dataset,table_name,gcs_base):
    client=bq_client()
    full=f"{source_project}.{source_dataset}.{table_name}"
    ts=dt.datetime.utcnow().strftime("%Y%m%d%H%M%S")
    gcs_path=f"{gcs_base}/dt={ts}"
    dest_uri=f"{gcs_path}/*.parquet"

    job_config=bigquery.job.ExtractJobConfig(destination_format=bigquery.DestinationFormat.PARQUET)

    print(f"exporting data from {full} to {gcs_path}")

    job=client.extract_table(full,dest_uri,job_config=job_config)
    job.result()
    print(f"exported data from {full} to {gcs_path}")
    return dest_uri

In [0]:
rows=fetch_eligible_table()

if not rows:
    print("No tables to process with active flag 1 and load flag 1")
else:
    for r in rows:
        t=r["table_name"]
        bq_to_gcs_status(t,"IN_PROGRESS")
        try:
            print(f"processing {t}")
            uri=export_table_to_gcs(r["source_project"],r["source_dataset"],t,r["gcs_path"])
            print(f"exported data from {t} to {uri}")
            bq_to_gcs_status(t,"COMPLETED")
        except Exception as e:
            bq_to_gcs_status(t,"FAILED",err=e)
            print(f"error processing {t} - {e}")


